# A first look at your results

Run this after the demo finishes. It opens the tables the pipeline wrote, shows how they fit
together, and makes the plots worth checking first.

Nothing here loads the big mask volumes, so it runs in a few seconds.

In [1]:
# cell 1
import sys
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from tifffile import imread
except ModuleNotFoundError as missing:
    # Either EASI-PASS is not installed in this environment, or Jupyter picked a
    # different one: it searches ~/.local/share/jupyter/kernels first, so a
    # leftover kernel can shadow the right one.
    raise SystemExit(
        f"'{missing.name}' is not installed in the Python running this notebook:\n"
        f"  {sys.executable}\n\n"
        "Either EASI-PASS is not installed there, in which case, from the repo\n"
        "with that environment active:\n"
        "  pip install -e . -c requirements.txt\n\n"
        "Or the notebook is on a different environment than you expect. Compare\n"
        "the path above with the one you installed into, and if they differ use\n"
        "Kernel > Change Kernel to switch.\n"
    ) from None

# Point this at any finished OUTPUT folder. The default is the demo run.
OUTPUT = Path("demo_pre_run/JS078_demo/OUTPUT")
if not OUTPUT.exists():
    OUTPUT = Path("demo") / OUTPUT          # if the notebook is run from the repo root
PLANE = "0"

assert OUTPUT.exists(), f"No OUTPUT folder at {OUTPUT.resolve()} - edit the path above"
print("reading", OUTPUT)          # relative, so a saved notebook carries no local paths

reading demo_pre_run/JS078_demo/OUTPUT


## 1. The main table

One row per cell in the reference FISH round. The gene columns hold that cell's intensity per
channel; the `twoP_` columns say which functional cell it turned out to be.

Two matchers ran, mask overlap and soma-print. They are never reconciled, so both
sets of columns are here and you set your own threshold.
`twoP_somaprint_confident` is always True or False, never blank: a pick that did
not clear its gate is still recorded.

In [2]:
# cell 2
merged = pd.read_csv(OUTPUT / f"MERGED/aligned_extracted_features/full_table_mean_twop_plane{PLANE}.csv")

print(f"{len(merged)} FISH cells x {len(merged.columns)} columns\n")
print("columns:", list(merged.columns))

genes = [g for g in merged.columns if g.startswith("mean_round_")]

# Each matcher gets its own mask. They run independently and are never reconciled.
by_iou = merged.twoP_iou_match.notna()
by_soma = merged.twoP_somaprint_confident.fillna(False).astype(bool)

merged.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'demo_pre_run/JS078_demo/OUTPUT/MERGED/aligned_extracted_features/full_table_mean_twop_plane0.csv'

### How the table is laid out

What each column holds, and how much of it is filled. The checks below confirm the table
is shaped the way the text above describes.

In [ ]:
# cell 2b
def column_group(c):
    """What a column is for, read from its name."""
    if c == "mask_id_main":
        return "identity", "this cell's mask id in the reference FISH round"
    if c == "plane":
        return "context", "the functional plane this table covers"
    if c in genes:
        rnd, probe = c.replace("mean_round_", "").split("_", 1)
        return f"probe (round {rnd})", f"mean {probe} intensity in this cell"
    if c.startswith("twoP_"):
        return "2P match", {
            "twoP_iou_match": "functional cell id, by mask overlap",
            "twoP_iou": "overlap fraction for that pair, 0 to 1",
            "twoP_somaprint_match": "functional cell id, by soma-print",
            "twoP_somaprint_confident": "whether that pick cleared its gate",
        }.get(c, "")
    if c.startswith("round_"):
        return "later round", "match into a later FISH round"
    return "other", ""


layout = pd.DataFrame([(c, *column_group(c), str(merged[c].dtype),
                        f"{merged[c].notna().mean():.0%}") for c in merged.columns],
                      columns=["column", "group", "what it holds", "dtype", "filled"])

print(f"{len(merged)} rows x {len(merged.columns)} columns, one row per FISH cell\n")
print(layout.to_string(index=False))

# The plots below assume this shape, so check it here.
checks = {
    "one row per cell (mask_id_main unique)": merged.mask_id_main.is_unique,
    "every probe column is numeric": all(pd.api.types.is_numeric_dtype(merged[g]) for g in genes),
    "no probe column is entirely blank": all(merged[g].notna().any() for g in genes),
    "the table covers a single plane": merged.plane.nunique() == 1,
}

# A FISH-only run leaves the twoP_ columns empty, so these two apply only
# once a plane was matched.
if merged.twoP_iou_match.notna().any():
    checks["twoP_somaprint_confident is never blank"] = merged.twoP_somaprint_confident.notna().all()
    checks["twoP_iou lies between 0 and 1"] = bool(merged.twoP_iou.dropna().between(0, 1).all())
print()
for label, ok in checks.items():
    print(f"  {'ok  ' if ok else 'BAD '} {label}")

## 2. Adding positions

The main table has no coordinates. Cell centroids live in the intensity table, which is long
format: one row per cell per channel. Join on `mask_id` to place every cell.

In [ ]:
# cell 4
inten = pd.read_csv(next((OUTPUT / "HCR/extract_intensities").glob("*_probs_intensities.csv")))
pos = inten.groupby("mask_id")[["X", "Y", "Z"]].first()

df = merged.merge(pos, left_on="mask_id_main", right_index=True, how="left")
print(f"{df.X.notna().sum()}/{len(df)} cells placed")
df[["mask_id_main", "X", "Y", "Z", "twoP_iou_match"]].head(3)

## 3. Did the alignment work?

The clearest check. Every functional cell, coloured by whether it found a partner in the FISH
volume. Unmatched cells scattered evenly is normal; a whole region of red means the alignment
missed there.

In [ ]:
# cell 5
mean_img = imread(OUTPUT / f"2P/cellpose/lowres_meanImg_C0_plane{PLANE}.tiff").astype(float)
masks_2p = imread(OUTPUT / f"2P/cellpose/lowres_meanImg_C0_plane{PLANE}_masks.tiff")

matched_ids = set(merged.twoP_iou_match.dropna().astype(int))
lut = np.zeros((int(masks_2p.max()) + 1, 4))
lut[np.isin(np.arange(len(lut)), list(matched_ids))] = [0.15, 0.75, 0.35, 0.7]
lut[[i for i in range(len(lut)) if i not in matched_ids]] = [0.90, 0.25, 0.25, 0.7]
lut[0] = 0

fig, (a, b) = plt.subplots(1, 2, figsize=(14, 5))
lo, hi = np.percentile(mean_img, [1, 99.5])
a.imshow(np.clip((mean_img - lo) / (hi - lo), 0, 1), cmap="gray")
a.imshow(lut[masks_2p]); a.set_xticks([]); a.set_yticks([])
a.set_title(f"{len(matched_ids)} matched (green), {int(masks_2p.max()) - len(matched_ids)} not (red)")

b.scatter(df.X, df.Y, s=1, c="0.85", linewidths=0, label=f"all FISH cells ({len(df)})")
b.scatter(df.X[by_iou], df.Y[by_iou], s=3, c="#268c46", linewidths=0, label=f"matched ({by_iou.sum()})")
b.set_aspect("equal"); b.invert_yaxis(); b.set_xlabel("X (px)"); b.set_ylabel("Y (px)")
b.set_title("Where the functional plane landed in the volume")
b.legend(frameon=False, markerscale=3, fontsize=9)
plt.tight_layout()

## 4. Each probe, in the cells you recorded from

Cell centroids, coloured by expression, for the cells that matched a functional cell. Each
probe is scaled to its own 5th-95th percentile so channels of very different brightness stay
comparable, and every FISH cell is drawn faint underneath for context.

Every cell is drawn at its XY position, so all depths are collapsed onto one plane.

The panels are the probes named in `PREFERRED` in the next cell, PV and SST here. Edit it
for your own panel. DAPI is a nuclear stain present in every round and GCAMP reports the
functional indicator, so neither says anything about cell type.

The FISH mask volume is still read here because the next section needs it, so give it about
fifteen seconds.

In [ ]:
# cell 6
from tifffile import imread as _imread

hcr_ref = sorted((OUTPUT / "HCR" / "cellpose").glob("HCR*_masks.tiff"))[0]
hcr_3d = _imread(hcr_ref)                        # kept, the next cell needs the z axis

matched_cells = df[by_soma]
PREFERRED = ("PV", "SST")            # edit for your own panel
informative = [g for g in genes if not any(s in g.upper() for s in ("DAPI", "GCAMP"))] or genes
probes = [g for g in informative if any(g.upper().endswith("_" + p) for p in PREFERRED)]

# A panel without those probes falls back to the channels with the most signal.
probes += [g for g in sorted(informative, key=lambda g: merged[g].median(), reverse=True)
           if g not in probes]
probes = probes[:2]
print(f"{len(matched_cells)} soma-print matched cells, {len(probes)} probes")

fig, axes = plt.subplots(1, len(probes), figsize=(5.2 * len(probes), 5.4), squeeze=False)
for ax, gene in zip(axes.ravel(), probes):
    v = matched_cells[gene].to_numpy()
    lo, hi = np.nanpercentile(v, [5, 95])
    # A sparse probe can have both percentiles at zero, which blanks the panel.
    if hi - lo < 1e-9:
        lo, hi = 0.0, float(np.nanmax(v))

    # One dot per cell. Grey is every FISH cell, colour is the matched subset.
    ax.scatter(df.X, df.Y, s=1.5, c="0.88", linewidths=0)
    order = np.argsort(np.nan_to_num(v))          # dim first, so bright lands on top
    sc = ax.scatter(matched_cells.X.to_numpy()[order],
                    matched_cells.Y.to_numpy()[order],
                    c=np.clip(v[order], lo, hi), s=14, cmap="viridis",
                    vmin=lo, vmax=hi, linewidths=0)
    ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(gene.replace("mean_round_", "round ") + "\nall depths collapsed",
                 fontsize=11)
    plt.colorbar(sc, ax=ax, fraction=0.046, label="5th-95th pct")
plt.tight_layout()

## 5. The matched masks, overlaid

Every soma-print-matched pair in the plane, drawn together. **Green is a functional cell,
magenta is its EASI-FISH partner, and white is where the two overlap.** A well-registered
plane is mostly white, with coloured fringes where the two outlines disagree.

The FISH volume is sampled on the surface the functional sheet sits on, so the two masks
are compared where they meet.

In [ ]:
# cell 7
from scipy.ndimage import binary_erosion

# The 2P volume is a 2D sheet draped through 3D: one non-zero z per (y, x). Its
# argmax over z is the surface the functional plane sits on, so sampling the
# FISH masks there compares the two where they meet.
twop_3d = _imread(OUTPUT / f"2P/registered/twop_plane{PLANE}_aligned_3d.tiff")
twop_lbl = twop_3d.max(axis=0)
zmap = twop_3d.argmax(axis=0)
del twop_3d

yy, xx = np.mgrid[0:zmap.shape[0], 0:zmap.shape[1]]
hcr_lbl = hcr_3d[zmap, yy, xx]
del hcr_3d

# Paired cells only: each side keeps just the labels soma-print matched.
paired = df[by_soma]
keep_2p = set(paired.twoP_somaprint_match.dropna().astype(int))
keep_hcr = set(paired.mask_id_main.astype(int))
tp = np.isin(twop_lbl, list(keep_2p))
hc = np.isin(hcr_lbl, list(keep_hcr))

# One-pixel erosion on the 2P side, as the figure panels do, so touching cells
# read as separate.
tp = binary_erosion(tp, np.ones((3, 3)))

rgb = np.zeros((*tp.shape, 3), dtype=np.float32)
rgb[..., 0] = hc          # magenta = EASI-FISH
rgb[..., 1] = tp          # green   = functional
rgb[..., 2] = hc          # 2P + FISH together -> white

ys, xs = np.nonzero(tp | hc)                      # crop to where there is anything
y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
view = rgb[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), facecolor="black")
axes[0].imshow(view, interpolation="nearest")
axes[0].set_title(f"{len(keep_2p)} matched pairs", color="white")

cy, cx = view.shape[0] // 2, view.shape[1] // 2
Z = 150
axes[1].imshow(view[cy - Z:cy + Z, cx - Z:cx + Z], interpolation="nearest")
axes[1].set_title(f"the middle {2*Z} px, same image", color="white")

for ax in axes:
    ax.axis("off"); ax.set_facecolor("black")
plt.tight_layout(rect=[0, 0.06, 1, 1])
fig.text(0.5, 0.02, "green = functional     magenta = EASI-FISH     white = overlap",
         color="white", ha="center", fontsize=11)

## Where to go from here

```python
matched = merged[merged.twoP_iou_match.notna()]   # cells with functional data
matched.nlargest(20, genes[1])                    # the brightest cells for one gene
matched.to_csv("my_matched_cells.csv")            # take it somewhere else
```

`twoP_iou_match` and `twoP_somaprint_match` both hold the functional cell's mask id, so that
is the key to join your own traces back on.

With more than one FISH round there are also `round_{R}_` columns per extra round. The gene
columns are joined on `round_{R}_hybrid_match`, and `round_{R}_iou_match` shows what overlap
alone would have said.